# Tax Lien Redemption Probability – Exploratory Data Analysis

## Research Question
**What is the probability that a tax lien will be redeemed within X months?**

This notebook performs exploratory data analysis and builds a baseline machine learning model using the **Home Credit Default Risk** dataset as a proxy for tax lien redemption behavior.

## Loading the Dataset

The dataset is stored as a compressed **7z archive** named `application_train.7z`.

To keep the repository size smaller and make the workflow reproducible, the notebook extracts the archive before loading the CSV file into pandas.

In [ ]:
# Install library for extracting .7z files
!pip install -q py7zr

In [ ]:
import os
import py7zr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

archive_path = "../data/application_train.7z"
extract_path = "../data"
csv_path = os.path.join(extract_path, "application_train.csv")

# Only extract if the CSV is not already present
if not os.path.exists(csv_path):
    with py7zr.SevenZipFile(archive_path, mode="r") as archive:
        archive.extractall(path=extract_path)
    print("Extraction complete.")
else:
    print("CSV already extracted.")

In [ ]:
# Load the dataset
df = pd.read_csv(csv_path)
df.head()

## Dataset Overview

I first checked the basic size and structure of the dataset to confirm that it loaded correctly and to understand the scale of the analysis.

In [ ]:
df.shape

In [ ]:
df.info()

## Target Variable Distribution

Before building any model, it is important to understand the balance of the target variable.

In this dataset:

- **0 = loan repaid** (used here as a proxy for tax lien redemption)
- **1 = loan default** (used here as a proxy for non-redemption)

This helps explain why ROC-AUC is a good evaluation metric for the baseline model.

In [ ]:
sns.set_style("whitegrid")

target_counts = df["TARGET"].value_counts(normalize=True).sort_index()

plt.figure(figsize=(6, 4))
sns.barplot(x=target_counts.index, y=target_counts.values)
plt.title("Target Variable Distribution")
plt.xlabel("Loan Default Indicator")
plt.ylabel("Proportion of Observations")
plt.xticks([0, 1], ["Repaid (0)", "Default (1)"])
plt.show()

## Missing Values and Duplicate Check

I checked for duplicate rows and reviewed the extent of missingness in the dataset before moving into feature engineering.

In [ ]:
duplicate_count = df.duplicated().sum()
duplicate_count

In [ ]:
missing_summary = df.isnull().mean().sort_values(ascending=False)
missing_summary.head(15)

## Feature Engineering

I created several ratio-based and transformed variables to better capture financial stress and repayment capacity.

In [ ]:
df["CREDIT_INCOME_RATIO"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
df["ANNUITY_INCOME_RATIO"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
df["GOODS_CREDIT_RATIO"] = df["AMT_GOODS_PRICE"] / df["AMT_CREDIT"]
df["EXT_SOURCE_MEAN"] = df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1)
df["YEARS_BIRTH"] = -df["DAYS_BIRTH"] / 365
df["YEARS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan).abs() / 365
df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)

## Correlation Analysis of Engineered Features

After creating the engineered variables, I looked at the correlation between the key financial features and the target variable. This helps show whether the engineered ratios capture useful repayment-risk signal.

In [ ]:
engineered_features = [
    "TARGET",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "GOODS_CREDIT_RATIO",
]

corr_matrix = df[engineered_features].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix of Key Financial Variables")
plt.show()

## Baseline Model – Logistic Regression

For the baseline model, I used logistic regression because it is a standard starting point for binary classification and provides an interpretable benchmark for later models.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

features = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "GOODS_CREDIT_RATIO",
    "EXT_SOURCE_MEAN",
    "YEARS_BIRTH",
    "YEARS_EMPLOYED",
    "DAYS_EMPLOYED_ANOM",
]

model_df = df[["TARGET"] + features].dropna()

X = model_df[features]
y = model_df["TARGET"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

pred_proba = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, pred_proba)
pr_auc = average_precision_score(y_test, pred_proba)

print(f"ROC-AUC: {roc_auc:.3f}")
print(f"PR-AUC: {pr_auc:.3f}")

## Initial Interpretation

This baseline result suggests that repayment behavior is meaningfully predictable using structured financial variables. Even though this is only a proxy dataset, the result supports the idea that a probability-based risk scoring approach is feasible for the broader tax lien project.